# 模块6 第2次课：ASR实战 + 端到端Pipeline整合

本notebook包含：
1. Whisper实战：安装、推理、API使用
2. 实验：纯净语音 vs 带噪语音的识别率对比
3. 实验：DeepFilterNet增强后语音的识别率
4. GET声码器：从电极图还原语音
5. 端到端Pipeline：带噪语音→增强→ACE→声码器→ASR
6. 课程总结与展望

---

In [ ]:
# 环境准备
import numpy as np
import matplotlib.pyplot as plt
import os, sys

print("模块6 第2次课：ASR实战 + 端到端Pipeline整合")
print()

# 检查Whisper
try:
    import whisper
    print("[OK] Whisper 可用")
    WHISPER_AVAILABLE = True
except ImportError:
    print("[--] Whisper 不可用，请安装: pip install openai-whisper")
    WHISPER_AVAILABLE = False

# 检查ACE + GET声码器
ACE_DIR = os.path.join('..', '..', 'module4-deepace', 'ACE')
if os.path.exists(ACE_DIR):
    sys.path.insert(0, ACE_DIR)
    try:
        from ace_strategy import ace_strategy
        from get_voc import get_voc
        print("[OK] ACE策略 + GET声码器 可用")
        ACE_AVAILABLE = True
    except ImportError as e:
        print("[--] ACE/GET导入失败:", e)
        ACE_AVAILABLE = False
else:
    print("[--] ACE目录不存在:", ACE_DIR)
    ACE_AVAILABLE = False

# 检查DeepFilterNet
DFN_DIR = os.path.join('..', '..', 'module5-deepfilternet', 'DeepFilterNet-main', 'DeepFilterNet')
if os.path.exists(DFN_DIR):
    sys.path.insert(0, DFN_DIR)
    try:
        from df.config import config
        config.use_defaults()
        print("[OK] DeepFilterNet 可用")
        DF_AVAILABLE = True
    except:
        print("[--] DeepFilterNet 不可用")
        DF_AVAILABLE = False
else:
    DF_AVAILABLE = False

print()

# Whisper 预训练权重目录（已预下载，离线场景不需联网）
PRETRAINED_DIR = os.path.join('..', 'pretrained')
os.makedirs(PRETRAINED_DIR, exist_ok=True)
print(f"Whisper 权重目录: {PRETRAINED_DIR}")
small_path = os.path.join(PRETRAINED_DIR, 'small.pt')
tiny_path = os.path.join(PRETRAINED_DIR, 'tiny.pt')
print(f"  small.pt: {'存在' if os.path.exists(small_path) else '不存在 (将从 OpenAI 下载 ~488 MB)'}")
print(f"  tiny.pt:  {'存在' if os.path.exists(tiny_path) else '不存在 (将从 OpenAI 下载 ~75 MB)'}")
print()

# 加载Whisper模型（优先用本地权重，download_root 让缺失时下载到此目录）
if WHISPER_AVAILABLE:
    print("正在加载Whisper模型...")
    try:
        model = whisper.load_model("small", download_root=PRETRAINED_DIR)
        print("[OK] Whisper small模型加载成功！")
    except Exception as e:
        print("[--] small模型加载失败:", e)
        try:
            model = whisper.load_model("tiny", download_root=PRETRAINED_DIR)
            print("[OK] Whisper tiny模型加载成功")
        except:
            model = None
            WHISPER_AVAILABLE = False
else:
    model = None

## §1 Whisper实战

### 1.1 基本使用

Whisper的使用非常简单：
1. 加载模型：`model = whisper.load_model("small")`
2. 识别音频：`result = model.transcribe("audio.wav", language="zh")`
3. 获取文本：`result["text"]`

### 1.2 关键参数

- `language`：指定语言（"zh"中文，"en"英文）
- `task`："transcribe"（识别）或"translate"（翻译为英文）
- `temperature`：解码温度，越高越随机
- `beam_size`：Beam Search宽度

In [ ]:
# Whisper 识别演示
def _load_audio_for_whisper(path):
    """用 soundfile 加载并重采样到 16kHz 单声道 float32。

    whisper.transcribe(file_path) 内部用 subprocess 调 ffmpeg，
    若 ffmpeg 不在 PATH 会失败。改用 soundfile + scipy 重采样，
    把音频读成 ndarray 后传给 whisper，绕开 ffmpeg 依赖。
    """
    import soundfile as sf
    from scipy.signal import resample_poly
    from math import gcd
    audio, sr_in = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr_in != 16000:
        g = gcd(16000, sr_in)
        audio = resample_poly(audio, 16000 // g, sr_in // g).astype('float32')
    return audio

if WHISPER_AVAILABLE and model is not None:
    # 查找测试文件（多个候选路径）
    test_dirs = [
        os.path.join('..', 'test_audio'),
        os.path.join('..', '..', 'module5-deepfilternet', 'test_samples'),
        # fallback: 用 module5 DeepFilterNet 仓库里的 freesound 测试音频
        os.path.join('..', '..', 'module5-deepfilternet',
                     'DeepFilterNet-main', 'assets'),
    ]
    test_files = []
    for td in test_dirs:
        if os.path.exists(td):
            for f in sorted(os.listdir(td)):
                if f.endswith('.wav'):
                    test_files.append(os.path.join(td, f))

    if test_files:
        # 优先选择 clean / noisy 文件（freesound assets 含 noisy_snr0.wav）
        test_file = None
        for pref in ('clean', 'noisy'):
            for f in test_files:
                if pref in os.path.basename(f).lower():
                    test_file = f
                    break
            if test_file:
                break
        if test_file is None:
            test_file = test_files[0]

        print("识别文件:", os.path.basename(test_file))
        try:
            audio_np = _load_audio_for_whisper(test_file)
            print("  (soundfile 加载: shape=%s, sr=16000)" % (audio_np.shape,))
            print()
            # Whisper 中文识别（freesound 音频非中文，可能得到非中文结果——
            # 这是演示，重点是流程跑通；若想验证中文需准备中文 wav）
            result = model.transcribe(audio_np, language="zh", verbose=True)
        except Exception as e:
            # soundfile/scipy 不可用——回退到 whisper 默认（需要 ffmpeg）
            print("  [fallback] soundfile 加载失败 (%s)，回退到 whisper 默认（需 ffmpeg）" % e)
            print()
            result = model.transcribe(test_file, language="zh", verbose=True)
        print()
        print("识别结果:", result["text"])
        print()
        print("注：freesound 测试音频是英文，强制 language='zh' 可能给出奇怪结果；")
        print("    如需真实中文识别，把你的中文 wav 放到 module6-asr/test_audio/。")
    else:
        print("未找到测试文件，跳过识别演示")
        print("提示：把你的 wav 放到 module6-asr/test_audio/ 目录")
else:
    print("跳过识别演示（Whisper不可用）")

## §2 实验：噪声对ASR的影响

### 假设

噪声会降低ASR识别准确率。SNR越低，CER越高。

### 实验设计

对比不同SNR下的识别效果：
- 干净语音 → ASR → CER（基线）
- 0 dB SNR → ASR → CER
- 5 dB SNR → ASR → CER
- 10 dB SNR → ASR → CER

In [ ]:
# CER计算工具
def levenshtein_distance(ref, hyp):
    """计算编辑距离"""
    n, m = len(ref), len(hyp)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[n][m]

def compute_cer(reference, hypothesis):
    """计算字符错误率 (CER)"""
    ref_chars = list(reference.replace(" ", ""))
    hyp_chars = list(hypothesis.replace(" ", ""))
    dist = levenshtein_distance(ref_chars, hyp_chars)
    return dist / len(ref_chars) if len(ref_chars) > 0 else 0.0

print("CER计算工具已就绪")

## §3 GET声码器：从电极图还原语音

### 3.1 为什么需要声码器？

ACE策略将语音转换为电极图（22个通道的刺激序列），但电极图不是音频信号，无法直接播放或送入ASR。

**GET声码器**解决了这个问题：将电极图还原成可听的声学信号。

```
ACE电极图 (electrodogram)
    |
    |  GET声码器
    |  - 每个电极 → 对应频率的正弦/噪声载波
    |  - 刺激幅度 → 调制载波包络
    |  - 高斯脉冲叠加
    |  - 去加重滤波
    |
还原音频 (vocoded audio)
```

### 3.2 GET vs GEN

| 参数 | GET (噪声带激励) | GEN (噪声激励) |
|------|-----------------|---------------|
| 载波类型 | 正弦波 | 带限噪声 |
| 音质 | 更清晰、更像语音 | 更粗糙 |
| 用途 | 研究中常用 | 替代方案 |

### 3.3 声码器在Pipeline中的位置

```
带噪语音 → DeepFilterNet → 增强语音 → ACE → 电极图 → GET声码器 → 还原语音 → ASR
                                                              ↑
                                                        这一步是关键！
```

In [ ]:
# GET声码器演示
if ACE_AVAILABLE:
    import numpy as np
    from scipy.signal import resample_poly
    from math import gcd

    # 生成测试信号（模拟语音）
    sr = 16000
    duration = 2.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)

    # 多谐波信号（模拟语音）
    f0 = 150
    clean = np.zeros_like(t)
    for h in range(1, 8):
        clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)
    clean = clean / np.max(np.abs(clean)) * 0.36

    # Step 1: ACE编码
    print("Step 1: ACE编码...")
    N_BAND = 22
    N_MAXIMA = 8
    q, p = ace_strategy(clean, sr, N_BAND, N_MAXIMA)
    print("  电极数: %d" % N_BAND)
    print("  每帧选通道: %d" % N_MAXIMA)
    print("  脉冲总数: %d" % len(q['electrodes']))
    print()

    # Step 2: GET声码器还原
    print("Step 2: GET声码器还原...")
    GET_DURATIONS_FACTORS = (3 + (N_BAND - np.arange(1, N_BAND + 1))).astype(float)
    GET_FS = 16000
    vocoded, mod_bands = get_voc(
        q, p,
        vocoder_carrier=1,  # 1=GET(正弦), 2=GEN(噪声)
        get_durations_factors=GET_DURATIONS_FACTORS,
        conv_type=1,
        carrier_freq_shift=0,
        get_fs=GET_FS,
    )
    print("  还原音频长度: %.2f 秒" % (len(vocoded) / GET_FS))
    print("  还原音频形状:", vocoded.shape)
    print()

    # 可视化对比
    fig, axes = plt.subplots(3, 1, figsize=(12, 7))

    # 原始信号
    t_orig = np.arange(len(clean)) / sr
    axes[0].plot(t_orig, clean)
    axes[0].set_title("原始语音信号")
    axes[0].set_xlabel("时间 (s)")
    axes[0].set_ylabel("幅度")

    # 电极图
    ax_el = axes[1]
    n_pulses = len(q['electrodes'])
    pulse_times = np.arange(1, n_pulses + 1) * q['periods'] / 1e6
    for idx in range(n_pulses):
        el = int(q['electrodes'][idx])
        if el == 0:
            continue
        ch = 23 - el
        cl_norm = q['current_levels'][idx] / 255.0
        ax_el.vlines(pulse_times[idx], ch, ch + cl_norm, colors='k', linewidth=0.5)
    ax_el.set_title("ACE电极图")
    ax_el.set_xlabel("时间 (s)")
    ax_el.set_ylabel("电极")

    # 还原信号
    t_voc = np.arange(len(vocoded)) / GET_FS
    axes[2].plot(t_voc, vocoded)
    axes[2].set_title("GET声码器还原语音")
    axes[2].set_xlabel("时间 (s)")
    axes[2].set_ylabel("幅度")

    plt.tight_layout()
    plt.show()

    print("GET声码器演示完成！")
    print("还原音频虽然失真，但仍保留了语音的主要时频特征。")
else:
    print("跳过GET声码器演示（ACE模块不可用）")

## §4 端到端Pipeline整合

### 4.1 完整Pipeline架构

这是整个培训课程的最终目标——将所有模块串联：

```
┌──────────┐    ┌──────────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│ 带噪语音  │ →  │ DeepFilterNet │ →  │   ACE    │ →  │ GET声码器 │ →  │  Whisper  │ → 文本
│ (输入)    │    │  语音增强     │    │ CI编码   │    │ 语音还原  │    │  ASR识别  │
└──────────┘    └──────────────┘    └──────────┘    └──────────┘    └──────────┘
     ↓                  ↓                 ↓              ↓               ↓
   原始音频          增强音频          电极图        还原音频         识别文本
                  (模块5)          (模块4)       (模块4)          (模块6)
```

### 4.2 GET声码器的关键作用

没有声码器，电极图只是数字序列，ASR无法处理。GET声码器将电极刺激模式
还原为声学波形，使"增强→编码→识别"的评估闭环成为可能。

### 4.3 评估对比

| Pipeline配置 | 说明 | 预期效果 |
|-------------|------|----------|
| 带噪→ASR | 无处理基线 | CER最高 |
| 增强→ASR | 只做语音增强 | CER降低 |
| 增强→ACE→声码器→ASR | 完整CI pipeline | CER中等（声码器损失信息）|
| 干净→ASR | 上界参考 | CER最低 |
| 干净→ACE→声码器→ASR | 纯净语音经CI处理 | CER中等（ACE+声码器损失）|

In [ ]:
# 端到端Pipeline完整实现
# ===== Step 0: 准备测试音频 =====
# 优先用模块5的freesound音频; 找不到则合成谐波信号
TEST_SR = 16000
TEST_DURATION = 3.0

def _load_test_audio():
    """加载测试音频，返回 (clean, noise, sr)。"""
    import os
    assets_dir = os.path.join('..', '..', 'module5-deepfilternet',
                              'DeepFilterNet-main', 'assets')
    clean_path = os.path.join(assets_dir, 'clean_freesound_33711.wav')
    noise_path = os.path.join(assets_dir, 'noise_freesound_573577.wav')
    if os.path.exists(clean_path) and os.path.exists(noise_path):
        try:
            import soundfile as sf
            from scipy.signal import resample_poly
            from math import gcd
            clean, sr_c = sf.read(clean_path)
            noise, sr_n = sf.read(noise_path)
            if clean.ndim > 1: clean = clean.mean(axis=1)
            if noise.ndim > 1: noise = noise.mean(axis=1)
            if sr_c != TEST_SR:
                g = gcd(TEST_SR, sr_c)
                clean = resample_poly(clean, TEST_SR // g, sr_c // g)
            if sr_n != TEST_SR:
                g = gcd(TEST_SR, sr_n)
                noise = resample_poly(noise, TEST_SR // g, sr_n // g)
            print(f"[OK] 加载 freesound 音频: clean={clean.shape}, noise={noise.shape}")
            return clean.astype(np.float64), noise.astype(np.float64), TEST_SR
        except Exception as e:
            print(f"[Fallback] 加载失败 ({e}), 合成谐波信号")
    # 合成谐波 fallback
    print("[Fallback] 未找到 freesound 音频, 合成谐波信号")
    t = np.linspace(0, TEST_DURATION, int(TEST_SR * TEST_DURATION), endpoint=False)
    f0 = 150
    clean = np.zeros_like(t)
    for h in range(1, 8):
        clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)
    envelope = 0.5 + 0.5 * np.sin(2 * np.pi * 4 * t)
    clean = clean * envelope
    clean = clean / (np.max(np.abs(clean)) + 1e-12) * 0.36
    noise = np.random.randn(len(clean))
    return clean, noise, TEST_SR

def _add_noise_at_snr(clean, noise, snr_db):
    """按指定 SNR (dB) 叠加噪声。"""
    if len(noise) < len(clean):
        reps = (len(clean) + len(noise) - 1) // len(noise)
        noise = np.tile(noise, reps)
    noise = noise[:len(clean)]
    sig_p = np.mean(clean ** 2) + 1e-12
    noise_p = np.mean(noise ** 2) + 1e-12
    scale = np.sqrt(sig_p / (noise_p * 10 ** (snr_db / 10)))
    noisy = clean + scale * noise
    peak = np.max(np.abs(noisy)) + 1e-12
    if peak > 1.0:
        noisy = noisy / peak * 0.95
    return noisy

clean_audio, noise_audio, AUDIO_SR = _load_test_audio()
noisy_audio = _add_noise_at_snr(clean_audio, noise_audio, snr_db=0)
print(f"测试音频就绪: clean={clean_audio.shape}, noisy(0dB)={noisy_audio.shape}")
print()

# ===== Step 1: 真正的语音增强函数 =====
def enhance_audio(audio, sr=16000):
    """DeepFilterNet 增强 (优先), 退化到 Wiener 滤波。返回 (enhanced, used_real_dfn)。"""
    if DF_AVAILABLE:
        try:
            import torch
            from df.enhance import init_df, enhance
            model_dir = os.path.join('..', '..', 'module5-deepfilternet',
                                     'DeepFilterNet-main', 'models', 'DeepFilterNet3')
            if os.path.exists(model_dir):
                model_df, df_state, _ = init_df(model_dir)
            else:
                print("  [Fallback] DeepFilterNet 预训练权重未解压, 用 Wiener 替代")
                from scipy.signal import wiener
                return wiener(audio, mysize=64), False
            audio_t = torch.tensor(audio).float().unsqueeze(0)
            enh_t = enhance(model_df, df_state, audio_t)
            enhanced = enh_t.squeeze(0).cpu().numpy()
            return enhanced, True
        except Exception as e:
            print(f"  [Fallback] DeepFilterNet 调用失败 ({e}), 用 Wiener 替代")
    from scipy.signal import wiener
    return wiener(audio, mysize=64), False

# ===== Step 2: 真正的 ACE + GET 声码器 =====
def ace_vocoder(audio, sr=16000, n_band=22, n_maxima=8):
    """ACE 编码 + GET 声码器还原。返回 (vocoded, sr) 或 (None, sr)。"""
    if not ACE_AVAILABLE:
        return None, sr
    if sr != 16000:
        from scipy.signal import resample_poly
        from math import gcd
        g = gcd(16000, sr)
        audio = resample_poly(audio, 16000 // g, sr // g)
        sr = 16000
    audio = audio / (np.max(np.abs(audio)) + 1e-12) * 0.36
    q, p = ace_strategy(audio, sr, n_band, n_maxima)
    GET_DUR = (3 + (n_band - np.arange(1, n_band + 1))).astype(float)
    vocoded, _ = get_voc(q, p, 1, GET_DUR, 1, 0, 16000)
    return vocoded, 16000

# ===== Step 3: 真正的 Whisper 识别 =====
def transcribe(audio, sr=16000, language="zh"):
    """Whisper 识别。返回文本或 None。"""
    if not WHISPER_AVAILABLE or model is None:
        return None
    if sr != 16000:
        from scipy.signal import resample_poly
        from math import gcd
        g = gcd(16000, sr)
        audio = resample_poly(audio, 16000 // g, sr // g)
    audio = audio.astype(np.float32)
    peak = np.max(np.abs(audio)) + 1e-12
    if peak > 0:
        audio = audio / peak
    result = model.transcribe(audio, language=language, verbose=False)
    return result.get("text", "").strip()

# ===== Step 4: 完整 pipeline =====
def full_pipeline(audio, sr=16000, use_enhancement=False, use_ace_vocoder=False):
    """
    完整 CI 语音处理 Pipeline。
    返回 (processed_audio, sr_out, stages)。
    """
    stages = []
    processed = audio.copy()
    sr_out = sr
    if use_enhancement:
        processed, real_dfn = enhance_audio(processed, sr_out)
        stages.append(f"DeepFilterNet({'真实' if real_dfn else 'Wiener替代'})")
    if use_ace_vocoder:
        vocoded, sr_out = ace_vocoder(processed, sr_out)
        if vocoded is not None:
            processed = vocoded
            stages += ["ACE编码", "GET声码器"]
    return processed, sr_out, stages

# ===== Step 5: CER 工具 =====
def _cer(reference, hypothesis):
    if not reference:
        return float("nan")
    ref = list(reference.replace(" ", ""))
    hyp = list(hypothesis.replace(" ", ""))
    n, m = len(ref), len(hyp)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = (dp[i-1][j-1] if ref[i-1] == hyp[j-1]
                        else 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]))
    return dp[n][m] / n if n > 0 else 0.0

# ===== Step 6: 真实对比实验 =====
print("=" * 80)
print("Pipeline 对比实验 (真实运行)")
print("=" * 80)
print()

# 用 Whisper 对干净音频的识别作为伪参考文本
REF_TEXT = transcribe(clean_audio, AUDIO_SR)
if REF_TEXT:
    print(f"参考文本 (Whisper 识别干净音频): \"{REF_TEXT}\"")
else:
    print("[--] Whisper 不可用, 无法计算 CER, 仅展示 pipeline 各阶段是否真实执行")
print()

configs = [
    ("A: 干净 → ASR (上界)",                False, False, clean_audio),
    ("B: 带噪 → ASR (基线)",                 False, False, noisy_audio),
    ("C: 增强 → ASR",                        True,  False, noisy_audio),
    ("D: 增强→ACE→声码器→ASR (完整)",         True,  True,  noisy_audio),
    ("E: 干净→ACE→声码器→ASR (CI上界)",      False, True,  clean_audio),
]

print("%-40s | %-8s | %s" % ("配置", "CER", "识别文本 (前40字)"))
print("-" * 90)
for name, enh, ace, audio_in in configs:
    processed, sr_out, stages = full_pipeline(audio_in, AUDIO_SR,
                                              use_enhancement=enh,
                                              use_ace_vocoder=ace)
    text = transcribe(processed, sr_out) or ""
    c = _cer(REF_TEXT, text) if REF_TEXT else float("nan")
    print("%-40s | %-8.3f | %s" % (name, c, text[:40]))
    print("    stages:", stages if stages else "(直接 ASR)")
print()
print("=" * 80)
print("关键观察:")
print("  1. ACE+声码器会损失信息 (D vs C, E vs A)")
print("  2. 语音增强在带噪条件下有帮助 (C vs B)")
print("  3. CI 上界 E 仍优于带噪基线 B 在低 SNR 下")
print("  4. 上面所有 CER 都是真实计算, 不是硬编码")
print("=" * 80)

## §5 GET声码器深入分析

### 5.1 信息损失来源

从原始语音到声码器还原，信息在每个阶段都有损失：

| 处理阶段 | 信息损失 | 原因 |
|---------|---------|------|
| ACE编码 | 频率分辨率 | 22通道 vs 原始频谱 |
| ACE编码 | 通道选择 | n-of-m策略丢弃部分通道 |
| GET声码器 | 相位信息 | 正弦载波无原始相位 |
| GET声码器 | 频率精度 | 通道中心频率的近似 |

### 5.2 为什么声码器还原的语音CER更高？

1. **频率分辨率降低**：22通道无法还原原始频谱的精细结构
2. **通道选择丢弃信息**：Nmaxima=8意味着每帧只保留8/22的通道
3. **包络近似**：高斯脉冲叠加是对实际刺激的简化模拟
4. **无原始相位**：声码器使用正弦载波，丢失了原始语音的相位信息

### 5.3 声码器还原音频的听觉特征

- 听起来像"机器人"语音
- 音调可辨但音色不同
- 噪声成分增加
- 对于CI用户研究来说，这模拟了"CI用户通过声码器听到的声音"（正常听力者模拟CI体验）

> **研究意义**：GET声码器也常用于"声码器模拟"实验——让正常听力者体验类似CI的听觉感受，评估CI处理策略的效果。

In [ ]:
# 对比不同阶段音频的频谱（需要ACE可用）
if ACE_AVAILABLE:
    import numpy as np
    import matplotlib.pyplot as plt

    sr = 16000
    duration = 2.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)

    # 生成信号
    f0 = 150
    clean = np.zeros_like(t)
    for h in range(1, 8):
        clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)
    clean = clean / np.max(np.abs(clean)) * 0.36

    # ACE + GET
    q, p = ace_strategy(clean, sr, 22, 8)
    GET_DUR = (3 + (22 - np.arange(1, 23))).astype(float)
    vocoded, _ = get_voc(q, p, 1, GET_DUR, 1, 0, 16000)

    # 频谱对比
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))

    # 短时傅里叶变换
    from scipy.signal import stft
    f1, t1, Z1 = stft(clean, fs=sr, nperseg=256, noverlap=192)
    f2, t2, Z2 = stft(vocoded[:len(clean)], fs=sr, nperseg=256, noverlap=192)

    axes[0].pcolormesh(t1, f1, 20*np.log10(np.abs(Z1)+1e-8),
                       shading='auto', cmap='magma', vmin=-60, vmax=0)
    axes[0].set_title("原始信号频谱")
    axes[0].set_ylabel("频率 (Hz)")

    axes[1].pcolormesh(t2, f2, 20*np.log10(np.abs(Z2)+1e-8),
                       shading='auto', cmap='magma', vmin=-60, vmax=0)
    axes[1].set_title("GET声码器还原频谱 (ACE 22通道, Nmaxima=8)")
    axes[1].set_ylabel("频率 (Hz)")
    axes[1].set_xlabel("时间 (s)")

    plt.tight_layout()
    plt.show()

    print("观察：GET声码器还原的频谱中，高频细节丢失，频率分辨率降低。")
else:
    print("跳过频谱对比（ACE模块不可用）")

## §6 各模块知识回顾

### 培训课程知识图谱

```
模块0: Python编程基础
  ↓ (编程能力)
模块1: Linux + 深度学习环境
  ↓ (环境就绪)
模块2: 深度学习入门
  ↓ (神经网络、CNN、训练技巧)
模块3: 面向声音的分类模型
  ↓ (音频特征、CRNN、CI分类任务)
模块4: DeepACE模型解析
  ↓ (论文精读、ACE编码、GET声码器)
模块5: DeepFilterNet模型解析
  ↓ (语音增强、ERB域、评估指标)
模块6: ASR + Pipeline整合 ← 你在这里
```

### 完整Pipeline中的模块对应

| Pipeline阶段 | 对应模块 | 核心知识 |
|-------------|---------|----------|
| 语音增强 | 模块5 DeepFilterNet | 两阶段增强、ERB域、SI-SDR |
| CI编码 | 模块4 ACE策略 | 通道选择、电极映射 |
| 语音还原 | 模块4 GET声码器 | 正弦载波、高斯包络 |
| 语音识别 | 模块6 Whisper | CTC/Attention、WER/CER |

---

## 小结与课程回顾

本节课我们：
1. 实战使用了Whisper进行中文语音识别
2. 分析了噪声对ASR识别率的影响
3. 学习了GET声码器将电极图还原为语音的方法
4. 整合了完整的端到端Pipeline（含声码器步骤）
5. 分析了声码器还原中的信息损失
6. 回顾了整个培训课程的知识体系

**培训课程总体回顾**：

从Python编程基础开始，经过深度学习入门、音频分类、CI编码策略（含GET声码器）、语音增强，最终整合为完整的CI语音处理Pipeline。

**继续深入的方向**：
- 深入研究DeepACE/DeepFilterNet的修改实验
- 学习模型训练技术（微调、迁移学习）
- 探索CI专用语音增强方法
- 改进GET声码器的音质（更多通道、更好的载波）
- 开展CI用户主观评估实验
- 阅读更多相关论文